# Jours 3-4-5 - Classification de la Satisfaction des Passagers

## Objectifs
- Construire un modèle de classification performant pour prédire la satisfaction des passagers
- Appliquer des techniques avancées d'ingénierie des caractéristiques
- Évaluer les performances sur le jeu de test

## Introduction

Dans ce notebook, vous allez développer un modèle de classification pour prédire si un passager sera satisfait ou non de son vol. Ce projet s'inscrit dans la mission d'AeroAnalytics visant à améliorer l'expérience client pour une compagnie aérienne internationale.

Vous êtes libre d'explorer différentes approches et techniques pour construire le meilleur modèle possible. Ce notebook vous servira de guide, mais n'hésitez pas à expérimenter et à ajouter vos propres idées !

In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import pickle
import sys
import os

# Visualisation avec Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn pour le machine learning
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Modèles de classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Ajouter le chemin pour importer utils.py
sys.path.append(os.path.abspath('./'))
from utils import results_predictions_satisfaction

# Pour afficher plus de colonnes dans les DataFrames
pd.set_option('display.max_columns', 30)

# Utilisation de plotly dans pandas
pd.options.plotting.backend = "plotly"

## 1. Chargement des données

Si votre ordinateur a des soucis pour faire tourner les modèles de ML, vous pouvez utiliser des versions plus petites des jeux de données. Pour cela, remplacer `train.csv` et `test.csv`, ci-dessous, par:
- `train_50.csv` et `test_50.csv` pour n'utiliser que 50% des données.
- `train_20.csv` et `test_20.csv` pour n'utiliser que 20% des données.

Par contre, utiliser moins de données a tendance à donner de moins bons résultats. Ainsi, lorsque vous avez votre modèle final, n'oubliez pas de remettre toutes les données et de recommencer le notebook.

In [ ]:
# Chargement des données d'entraînement
train_df = pd.read_csv('../../data/passenger_satisfaction/train.csv')

# Chargement des données de test
test_df = pd.read_csv('../../data/passenger_satisfaction/test.csv')

# Affichage des premières lignes
train_df.head()

In [ ]:
# Affichage des premières lignes
test_df.head()

In [ ]:
# Vérification des dimensions
print(f"Dimensions du jeu d'entraînement : {train_df.shape}")
print(f"Dimensions du jeu de test : {test_df.shape}")

In [ ]:
# Vérification des valeurs manquantes pour train et test
missing_values_train = train_df.isnull().sum()
missing_values_test = test_df.isnull().sum()
missing_values_pct_train = (missing_values_train / len(train_df)) * 100
missing_values_pct_test = (missing_values_test / len(test_df)) * 100

# Création d'un DataFrame pour visualiser les valeurs manquantes
missing_df = pd.DataFrame({
    'Train - Nombre': missing_values_train,
    'Train - %': missing_values_pct_train,
    'Test - Nombre': missing_values_test, 
    'Test - %': missing_values_pct_test
})

# Affichage des colonnes avec des valeurs manquantes
missing_df[
    (missing_df['Train - Nombre'] > 0) | 
    (missing_df['Test - Nombre'] > 0)
].sort_values('Train - Nombre', ascending=False)

## 2. Ingénierie des caractéristiques

L'ingénierie des caractéristiques est une étape cruciale pour améliorer les performances des modèles de machine learning. Dans cette section, nous allons explorer différentes techniques pour créer de nouvelles caractéristiques pertinentes à partir des données existantes.

### 2.1 Traitement des valeurs manquantes

Commençons par traiter les valeurs manquantes dans nos données.

In [ ]:
# Exemple de traitement des valeurs manquantes
# Pour les variables numériques, nous pouvons utiliser la médiane
# Pour les variables catégorielles, nous pouvons utiliser le mode (valeur la plus fréquente)

# Copie des DataFrames pour ne pas modifier les originaux
train_processed = train_df.copy()
test_processed = test_df.copy()

# Remplacement des valeurs manquantes pour 'Arrival Delay in Minutes'
train_processed['Arrival Delay in Minutes'] = train_processed['Arrival Delay in Minutes'].fillna(train_processed['Arrival Delay in Minutes'].median())
test_processed['Arrival Delay in Minutes'] = test_processed['Arrival Delay in Minutes'].fillna(test_processed['Arrival Delay in Minutes'].median())

# Vérification
print("Valeurs manquantes après traitement :")
print(train_processed.isnull().sum()[train_processed.isnull().sum() > 0])

### 2.2 Création de nouvelles caractéristiques

Créons maintenant de nouvelles caractéristiques qui pourraient être utiles pour notre modèle.

In [ ]:
# Exemple 1 : Création d'une caractéristique pour le retard total
train_processed['Total Delay'] = train_processed['Departure Delay in Minutes'] + train_processed['Arrival Delay in Minutes']
test_processed['Total Delay'] = test_processed['Departure Delay in Minutes'] + test_processed['Arrival Delay in Minutes']

# Exemple 2 : Catégorisation de l'âge
def categorize_age(age):
    if age < 18:
        return 'Jeune'
    elif age < 35:
        return 'Jeune adulte'
    elif age < 55:
        return 'Adulte'
    else:
        return 'Senior'

train_processed['Age Category'] = train_processed['Age'].apply(categorize_age)
test_processed['Age Category'] = test_processed['Age'].apply(categorize_age)

# Exemple 3 : Score moyen de satisfaction des services
service_columns = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
                   'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
                   'Inflight entertainment', 'On-board service', 'Leg room service',
                   'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness']

train_processed['Average Service Rating'] = train_processed[service_columns].mean(axis=1)
test_processed['Average Service Rating'] = test_processed[service_columns].mean(axis=1)

# Affichage des nouvelles caractéristiques
train_processed[['Total Delay', 'Age Category', 'Average Service Rating']].head()

### 2.3 Transformation des variables catégorielles

Transformons les variables catégorielles en variables numériques pour qu'elles puissent être utilisées par nos modèles.

In [ ]:
# Exemple d'encodage one-hot pour les variables catégorielles
categorical_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'Age Category']

# Récupération de toutes les valeurs uniques des ensembles d'entraînement et de test
all_categories = {}
for feature in categorical_features:
    train_categories = set(train_processed[feature].unique())
    test_categories = set(test_processed[feature].unique())
    all_categories[feature] = train_categories.union(test_categories)

# Création de DataFrames vides avec toutes les catégories possibles
train_encoded = train_processed.copy()
test_encoded = test_processed.copy()

# Encodage de chaque variable catégorielle
for feature in categorical_features:
    # Création des variables indicatrices pour toutes les catégories possibles
    train_dummies = pd.get_dummies(train_processed[feature], prefix=feature, drop_first=True)
    test_dummies = pd.get_dummies(test_processed[feature], prefix=feature, drop_first=True)
    
    # Ajout des colonnes manquantes dans chaque ensemble
    missing_in_train = set(test_dummies.columns) - set(train_dummies.columns)
    missing_in_test = set(train_dummies.columns) - set(test_dummies.columns)
    
    for col in missing_in_train:
        train_dummies[col] = 0
    for col in missing_in_test:
        test_dummies[col] = 0
        
    # Assurance du même ordre des colonnes
    train_dummies = train_dummies.reindex(columns=sorted(train_dummies.columns))
    test_dummies = test_dummies.reindex(columns=sorted(test_dummies.columns))
    
    # Suppression de la colonne d'origine et ajout des colonnes encodées
    train_encoded = train_encoded.drop(feature, axis=1)
    test_encoded = test_encoded.drop(feature, axis=1)
    train_encoded = pd.concat([train_encoded, train_dummies], axis=1)
    test_encoded = pd.concat([test_encoded, test_dummies], axis=1)

# Vérification des colonnes après encodage
print(f"Nombre de colonnes après encodage : {train_encoded.shape[1]}")
print("Nouvelles colonnes créées :")
new_columns = [col for col in train_encoded.columns if col not in train_df.columns]
print(new_columns[:10])  # Affichage des 10 premières nouvelles colonnes

### 2.4 Normalisation des variables numériques

Normalisons les variables numériques pour améliorer les performances de certains modèles.

In [ ]:
# Exemple de normalisation avec StandardScaler
numerical_features = ['Age', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Total Delay', 'Average Service Rating'] + service_columns

# Création d'un scaler
scaler = StandardScaler()

# Application du scaler aux caractéristiques numériques
train_encoded[numerical_features] = scaler.fit_transform(train_encoded[numerical_features])
test_encoded[numerical_features] = scaler.transform(test_encoded[numerical_features])

# Vérification des résultats
train_encoded[numerical_features].describe()

### 2.5 Préparation des données pour l'entraînement

Préparons maintenant nos données pour l'entraînement des modèles.

In [ ]:
# Séparation des caractéristiques et de la variable cible
X_train = train_encoded.drop(['Satisfaction', 'ID'], axis=1)
y_train = train_encoded['Satisfaction']

# Transform the y_train to a 0-1 variable
y_train = y_train.map({'satisfied': 1, 'neutral or dissatisfied': 0})

# Préparation des données de test
X_test = test_encoded.drop(['ID'], axis=1)  # Notez que le jeu de test n'a pas la colonne 'Satisfaction'

# Vérification des dimensions
print(f"Dimensions de X_train : {X_train.shape}")
print(f"Dimensions de y_train : {y_train.shape}")
print(f"Dimensions de X_test : {X_test.shape}")

## 3. Modélisation et évaluation

### 3.1 Tester si notre modèle fonctionne bien

Avant de tester si notre modèle fonctionne bien, il faut le définir.

In [ ]:
model = RandomForestClassifier(n_estimators=50)

model

### 3.2 Train-test split

Si on veut faire un test rapide sans entraîner le modèle plusieurs fois, on peut découper nos données en train et validation.

In [ ]:
# On découpe les données en ensembles d'entraînement et de validation
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Entraînement du modèle
model.fit(X_train_split, y_train_split)

# Prédictions sur l'ensemble de validation
y_pred = model.predict(X_val)

print(f"L'accuracy moyenne est de {accuracy_score(y_val, y_pred)}")

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_pred=y_pred, y_true=y_val)

# Create labels
labels = ['neutral or dissatisfied', 'satisfied']

# Create heatmap using plotly
fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels,
    y=labels,
    text=cm,
    texttemplate="%{text}",
    textfont={"size": 20},
    hoverongaps=False,
    colorscale='RdBu'))

# Update layout
fig.update_layout(
    title='Confusion Matrix',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    width=600,
    height=600
)

fig.show()

### 3.3 Validation croisée

Si on veut faire un test complet, on peut faire une validation croisée.

In [ ]:
# On utilise la fonction cross_val_score pour faire une validation croisée
accuracies = cross_val_score(model, X_train, y_train, cv=5)

print(f"L'accuracy moyenne est de {accuracies.mean()}")

### 3.4 Évaluation sur le jeu de test

Si vous êtes contents avec votre modèle, vous pouvez l'entraîner sur toutes les données d'entraînement et faire des prédictions sur le jeu de test.

In [15]:
# Entraînement du modèle sur l'ensemble des données d'entraînement (celui qui a été défini plus haut)
model.fit(X_train, y_train)

# Prédictions sur le jeu de test
y_pred = model.predict(X_test)

# Transform y_pred into satisfied or dissatisfied
y_pred = np.where(y_pred == 1, 'satisfied', 'neutral or dissatisfied')

In [ ]:
# Utilisation de la fonction results_predictions_satisfaction pour évaluer les performances
# Cette fonction va charger les vraies étiquettes du jeu de test et calculer le score F1
f1_test = results_predictions_satisfaction(y_pred)
print(f"Score F1 sur le jeu de test : {f1_test:.5f}")